Learning rate for stage 2 changed from the first model

In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
import os
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import pearsonr

2025-03-04 11:27:13.872273: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-03-04 11:27:14.218798: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-03-04 11:27:14.218856: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-03-04 11:27:14.318905: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-03-04 11:27:14.455682: I tensorflow/core/platform/cpu_feature_guar

In [2]:
# load teh models required
frozen_model = tf.keras.models.load_model('models/pre_trained_CNN_architecture_CNN_LSTM_frozen_adjustment_1.keras')
finetuned_model = tf.keras.models.load_model('models/pre_trained_CNN_architecture_CNN_LSTM_finetuned_adjustment_1.keras')

2025-03-04 11:27:34.768259: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 31134 MB memory:  -> device: 0, name: Tesla V100S-PCIE-32GB, pci bus id: 0000:86:00.0, compute capability: 7.0


In [3]:
# load a test image to get the height and width information
file_path = 'All_data/block_0103/all_np_files/Block0103_2020_08_26.npy'

loaded_test_image = np.load(file_path)

print(loaded_test_image.shape)
image_height = loaded_test_image.shape[0]
image_width = loaded_test_image.shape[1]
print(image_height, image_width)

(768, 1024, 3)
768 1024


In [4]:
def get_all_preds_in_test_time_series(block_name, main_data_folder, model):
    # get the path to the block
    path_to_block = os.path.join(main_data_folder, block_name)
    # load the numpy file with the feature sub-widows
    test_features = np.load(os.path.join(path_to_block, [file for file in os.listdir(path_to_block) if file[:9] == 'subwindow'][0]))
    # get the predicted values
    all_predicted_values = model.predict(test_features)

    # return the predicted values
    return(all_predicted_values)

In [5]:
def prediction_on_test_data(pred_values, image_height, image_width, stride = 8, kernel_size = 32):
    # density map
    Density_map = np.zeros((image_height, image_width))

    # counts map
    counts_map = np.zeros((image_height, image_width))
    
    # now, for every window, we will keep adding the values together and also add the counts
    counter = 0
#     need a counter to move into each predicted value in the pred values list
    for ii in range(0, image_height, stride):
        for jj in range(0, image_width, stride):
#         operations for density map
#             get the window of interest
            new_window = Density_map[ii:ii + kernel_size,jj:jj+kernel_size]
#     fill each with the value c_k
            counts_window = np.full((new_window.shape[0], new_window.shape[1]), pred_values[counter])
#     get the shapes of this new window
            cw_height = counts_window.shape[0]
            cw_width = counts_window.shape[1]
#         Do c_k/r_2
            counts_window_new = counts_window/(cw_height*cw_width)
#     This is the value in the window now
            value_window = counts_window_new
#     place the values in the corrsponding area of the density map
            Density_map[ii:ii + kernel_size,jj:jj+kernel_size] = new_window + value_window

#         Let's now focus on capturing the counts of the windows
            new_window_c = counts_map[ii:ii + kernel_size,jj:jj+kernel_size]
#     get the counts area
            count = np.ones((new_window_c.shape[0], new_window_c.shape[1]))
#     keep adding the counts to reflect the addition of densities
            counts_map[ii:ii + kernel_size,jj:jj+kernel_size] = new_window_c + count
#     increase the counter
            counter = counter + 1
            
#         get the normalized count
    normalized_counts = np.divide(Density_map, counts_map)
    
#     entire count on the test set
    pred_on_test = np.sum(normalized_counts)
    
#     return the predicted value
    return(pred_on_test, normalized_counts)


In [6]:
def get_final_forecasted_and_true_values(preds_from_model, im_height, im_weight, stride, kernel_size, csv_file_name, block_name):
    final_preds_list = []
    for i in range(7):
        preds_per_image = prediction_on_test_data(preds_from_model[:,i], im_height, im_weight, stride , kernel_size)
        final_preds_list.append(preds_per_image[0])
    # make this list  a dataframe
    preds_df = pd.DataFrame(final_preds_list, columns = ['Forecasted_value'])
    
    # Where do we have the true values?
    true_val_location = 'All_data/test_true_counts'
    true_value_file = pd.read_csv(os.path.join(true_val_location, csv_file_name))
    
    # compute the mae
    mae_value = mean_absolute_error(true_value_file[['True_count']], preds_df[['Forecasted_value']])
    # compute the rmse
    rmse_value = np.sqrt(mean_squared_error(true_value_file[['True_count']], preds_df[['Forecasted_value']]))
    # pearsonr
    pearson_value = pearsonr(np.array(true_value_file[['True_count']]).reshape(-1), np.array(preds_df[['Forecasted_value']]).reshape(-1))
    # r2score
    r2score_value = r2_score(true_value_file[['True_count']], preds_df[['Forecasted_value']])
    # attach the true and the forecasted values together
    final_df = pd.concat((true_value_file, preds_df), axis = 1)
    # final df location
    final_loc = 'All_data/test_predicted_counts'
    # save this file
    final_df.to_csv(os.path.join(final_loc, block_name + '.csv'), index = False)
    all_metrics = [mae_value, rmse_value, pearson_value, r2score_value]

    return(final_preds_list, all_metrics, final_df)

Block 0103

Predictions wth the frozen model

In [7]:
# first get the predictions
frozen_preds_block_0103 = get_all_preds_in_test_time_series('block_0103', 'All_data', frozen_model)

2025-03-04 11:27:55.032847: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8907


384/384 [==============================] - 5s 5ms/step


In [8]:
frozen_preds_block_0103.shape

(12288, 7)

In [9]:
frozen_final_forecasts_block_0103 = get_final_forecasted_and_true_values(frozen_preds_block_0103, image_height, image_width, 8, 32, 'true_counts_blk_0103.csv', 'frozen_block_0103_adjustment_1')

In [10]:
frozen_normalized_forecasts_block_0103 = frozen_final_forecasts_block_0103[0]

In [11]:
print(frozen_normalized_forecasts_block_0103)

[49.06633365843055, 64.51797347073361, 76.82281779175133, 59.44814773824408, 41.92926127564473, 34.25095692614396, 14.639333703649173]


In [12]:
mae_frozen_block_0103 = frozen_final_forecasts_block_0103[1]
mae_frozen_block_0103

[18.127749043573026,
 21.079321476514934,
 PearsonRResult(statistic=0.5928206335755777, pvalue=0.16068043328359255),
 -15.594932851831498]

In [13]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0103[2]
frozen_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0103_2020_08_26,40,40.000661,49.066334
1,Block0103_2020_08_27,39,39.000001,64.517973
2,Block0103_2020_08_28,41,41.000000,76.822818
3,Block0103_2020_08_31,31,31.000000,59.448148
4,Block0103_2020_09_02,32,32.000000,41.929261
5,Block0103_2020_09_07,40,40.002086,34.250957
6,Block0103_2020_09_16,27,27.000176,14.639334


Predictions with the finetuned model

In [14]:
# first get the predictions
finetuned_preds_block_0103 = get_all_preds_in_test_time_series('block_0103', 'All_data', finetuned_model)

384/384 [==============================] - 2s 5ms/step


In [15]:
finetuned_preds_block_0103.shape

(12288, 7)

In [16]:
finetuned_final_forecasts_block_0103 = get_final_forecasted_and_true_values(finetuned_preds_block_0103, image_height, image_width, 8, 32, 'true_counts_blk_0103.csv', 'finetuned_block_0103_adjustment_1')

In [17]:
finetuned_normalized_forecasts_block_0103 = finetuned_final_forecasts_block_0103[0]

In [18]:
print(finetuned_normalized_forecasts_block_0103)

[17.479747792404925, 37.80447738473516, 60.88717872890347, 34.544559904781224, 25.192022447726025, 28.995112130616135, 10.896951214485549]


In [19]:
mae_finetuned_block_0103 = finetuned_final_forecasts_block_0103[1]
mae_finetuned_block_0103

[11.580489666245272,
 13.85341171021921,
 PearsonRResult(statistic=0.5405526986570436, pvalue=0.21029913914182724),
 -6.167632457796567]

In [20]:
finetuned_true_forecasted_df = finetuned_final_forecasts_block_0103[2]
finetuned_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0103_2020_08_26,40,40.000661,17.479748
1,Block0103_2020_08_27,39,39.000001,37.804477
2,Block0103_2020_08_28,41,41.000000,60.887179
3,Block0103_2020_08_31,31,31.000000,34.544560
4,Block0103_2020_09_02,32,32.000000,25.192022
5,Block0103_2020_09_07,40,40.002086,28.995112
6,Block0103_2020_09_16,27,27.000176,10.896951


Block 0104

Predictions wth the frozen model

In [21]:
# first get the predictions
frozen_preds_block_0104 = get_all_preds_in_test_time_series('block_0104', 'All_data', frozen_model)

384/384 [==============================] - 2s 5ms/step


In [22]:
frozen_preds_block_0104.shape

(12288, 7)

In [23]:
frozen_final_forecasts_block_0104 = get_final_forecasted_and_true_values(frozen_preds_block_0104, image_height, image_width, 8, 32, 'true_counts_blk_0104.csv', 'frozen_block_0104_adjustment_1')

In [24]:
frozen_normalized_forecasts_block_0104 = frozen_final_forecasts_block_0104[0]

In [25]:
print(frozen_normalized_forecasts_block_0104)

[38.56588091631907, 40.37951701666787, 54.80401907117994, 51.291393161238766, 31.773565930529113, 24.0871072195609, 12.886474559886459]


In [26]:
mae_frozen_block_0104 = frozen_final_forecasts_block_0104[1]
mae_frozen_block_0104

[12.470523207918452,
 13.173824418081297,
 PearsonRResult(statistic=0.27701682081994433, pvalue=0.5475561809615863),
 -6.330976586313203]

In [27]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0104[2]
frozen_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0104_2020_08_26,33,33.000000,38.565881
1,Block0104_2020_08_27,30,30.000000,40.379517
2,Block0104_2020_08_28,39,39.000001,54.804019
3,Block0104_2020_08_31,40,40.000000,51.291393
4,Block0104_2020_09_02,41,40.998810,31.773566
5,Block0104_2020_09_07,42,42.169009,24.087107
6,Block0104_2020_09_16,30,30.005317,12.886475


Predictions with the finetuned model

In [28]:
# first get the predictions
finetuned_preds_block_0104 = get_all_preds_in_test_time_series('block_0104', 'All_data', finetuned_model)

384/384 [==============================] - 2s 5ms/step


In [29]:
finetuned_preds_block_0104.shape

(12288, 7)

In [30]:
finetuned_final_forecasts_block_0104 = get_final_forecasted_and_true_values(finetuned_preds_block_0104, image_height, image_width, 8, 32, 'true_counts_blk_0104.csv', 'finetuned_block_0104_adjustment_1')

In [31]:
finetuned_normalized_forecasts_block_0104 = finetuned_final_forecasts_block_0104[0]

In [32]:
print(finetuned_normalized_forecasts_block_0104)

[9.05623753596895, 18.129728049706802, 42.30699661156866, 18.23747923782806, 12.874124878436458, 15.290655569462919, 5.780747433163015]


In [33]:
mae_finetuned_block_0104 = finetuned_final_forecasts_block_0104[1]
mae_finetuned_block_0104

[19.99114627242892,
 21.67874792042013,
 PearsonRResult(statistic=0.3546864512599747, pvalue=0.4350147467710006),
 -18.85210125729214]

In [34]:
finetuned_true_forecasted_df = finetuned_final_forecasts_block_0104[2]
finetuned_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0104_2020_08_26,33,33.000000,9.056238
1,Block0104_2020_08_27,30,30.000000,18.129728
2,Block0104_2020_08_28,39,39.000001,42.306997
3,Block0104_2020_08_31,40,40.000000,18.237479
4,Block0104_2020_09_02,41,40.998810,12.874125
5,Block0104_2020_09_07,42,42.169009,15.290656
6,Block0104_2020_09_16,30,30.005317,5.780747


Block 0105

Predictions wth the frozen model

In [35]:
# first get the predictions
frozen_preds_block_0105 = get_all_preds_in_test_time_series('block_0105', 'All_data', frozen_model)

384/384 [==============================] - 2s 5ms/step


In [36]:
frozen_preds_block_0105.shape

(12288, 7)

In [37]:
frozen_final_forecasts_block_0105 = get_final_forecasted_and_true_values(frozen_preds_block_0105, image_height, image_width, 8, 32, 'true_counts_blk_0105.csv', 'frozen_block_0105_adjustment_1')

In [38]:
frozen_normalized_forecasts_block_0105 = frozen_final_forecasts_block_0105[0]

In [39]:
print(frozen_normalized_forecasts_block_0105)

[43.189854337481364, 52.451645140624976, 65.67133171598833, 54.729715692500285, 34.29217236829534, 28.84676836527873, 12.72605334751926]


In [40]:
mae_frozen_block_0105 = frozen_final_forecasts_block_0105[1]
mae_frozen_block_0105

[7.882507543643089,
 8.458900411343356,
 PearsonRResult(statistic=0.9259930494396733, pvalue=0.0027491335792367844),
 0.2450265262096869]

In [41]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0105[2]
frozen_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0105_2020_08_26,40,40.000001,43.189854
1,Block0105_2020_08_27,46,46.002743,52.451645
2,Block0105_2020_08_28,58,58.000696,65.671332
3,Block0105_2020_08_31,41,41.000032,54.729716
4,Block0105_2020_09_02,41,41.001190,34.292172
5,Block0105_2020_09_07,36,36.000022,28.846768
6,Block0105_2020_09_16,23,23.000000,12.726053


Predictions with the finetuned model

In [42]:
# first get the predictions
finetuned_preds_block_0105 = get_all_preds_in_test_time_series('block_0105', 'All_data', finetuned_model)

384/384 [==============================] - 2s 5ms/step


In [43]:
finetuned_preds_block_0105.shape

(12288, 7)

In [44]:
finetuned_final_forecasts_block_0105 = get_final_forecasted_and_true_values(finetuned_preds_block_0105, image_height, image_width, 8, 32, 'true_counts_blk_0105.csv', 'finetuned_block_0105_adjustment_1')

In [45]:
finetuned_normalized_forecasts_block_0105 = finetuned_final_forecasts_block_0105[0]

In [46]:
print(finetuned_normalized_forecasts_block_0105)

[10.724875650229782, 23.777731098407457, 50.46982199713665, 23.853859522538183, 16.840287282162837, 20.175253681153816, 6.703524698100258]


In [47]:
mae_finetuned_block_0105 = finetuned_final_forecasts_block_0105[1]
mae_finetuned_block_0105

[18.922092295753004,
 20.00444692271676,
 PearsonRResult(statistic=0.8804879126250599, pvalue=0.008883740895530366),
 -3.2223766015301063]

In [48]:
finetuned_true_forecasted_df = finetuned_final_forecasts_block_0105[2]
finetuned_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0105_2020_08_26,40,40.000001,10.724876
1,Block0105_2020_08_27,46,46.002743,23.777731
2,Block0105_2020_08_28,58,58.000696,50.469822
3,Block0105_2020_08_31,41,41.000032,23.853860
4,Block0105_2020_09_02,41,41.001190,16.840287
5,Block0105_2020_09_07,36,36.000022,20.175254
6,Block0105_2020_09_16,23,23.000000,6.703525


Block 0106

Predictions wth the frozen model

In [49]:
# first get the predictions
frozen_preds_block_0106 = get_all_preds_in_test_time_series('block_0106', 'All_data', frozen_model)

384/384 [==============================] - 2s 5ms/step


In [50]:
frozen_preds_block_0106.shape

(12288, 7)

In [51]:
frozen_final_forecasts_block_0106 = get_final_forecasted_and_true_values(frozen_preds_block_0106, image_height, image_width, 8, 32, 'true_counts_blk_0106.csv', 'frozen_block_0106_adjustment_1')

In [52]:
frozen_normalized_forecasts_block_0106 = frozen_final_forecasts_block_0106[0]

In [53]:
print(frozen_normalized_forecasts_block_0106)

[41.99758534083471, 49.46053449751764, 63.00454466749016, 55.539966060606986, 34.77158541322729, 28.79074489059045, 13.267994250105808]


In [54]:
mae_frozen_block_0106 = frozen_final_forecasts_block_0106[1]
mae_frozen_block_0106

[14.167472287503708,
 15.73184609356266,
 PearsonRResult(statistic=0.10532062248894934, pvalue=0.8221921575218379),
 -19.76551043504383]

In [55]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0106[2]
frozen_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0106_2020_08_26,39,38.999667,41.997585
1,Block0106_2020_08_27,39,38.999989,49.460534
2,Block0106_2020_08_28,45,45.000000,63.004545
3,Block0106_2020_08_31,40,39.986887,55.539966
4,Block0106_2020_09_02,43,42.999997,34.771585
5,Block0106_2020_09_07,48,47.996502,28.790745
6,Block0106_2020_09_16,38,38.000000,13.267994


Predictions with the finetuned model

In [56]:
# first get the predictions
finetuned_preds_block_0106 = get_all_preds_in_test_time_series('block_0106', 'All_data', finetuned_model)

384/384 [==============================] - 2s 5ms/step


In [57]:
finetuned_preds_block_0106.shape

(12288, 7)

In [58]:
finetuned_final_forecasts_block_0106 = get_final_forecasted_and_true_values(finetuned_preds_block_0106, image_height, image_width, 8, 32, 'true_counts_blk_0106.csv', 'finetuned_block_0106_adjustment_1')

In [59]:
finetuned_normalized_forecasts_block_0106 = finetuned_final_forecasts_block_0106[0]

In [60]:
print(finetuned_normalized_forecasts_block_0106)

[13.288161620464354, 26.48880747809987, 48.84632232365322, 27.90284197505764, 19.608616603852017, 22.70982604870005, 8.905892998773044]


In [61]:
mae_finetuned_block_0106 = finetuned_final_forecasts_block_0106[1]
mae_finetuned_block_0106

[18.848882228386604,
 20.743263198117784,
 PearsonRResult(statistic=0.4826759129818734, pvalue=0.2726008842374129),
 -35.10250931029623]

In [62]:
finetuned_true_forecasted_df = finetuned_final_forecasts_block_0106[2]
finetuned_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0106_2020_08_26,39,38.999667,13.288162
1,Block0106_2020_08_27,39,38.999989,26.488807
2,Block0106_2020_08_28,45,45.000000,48.846322
3,Block0106_2020_08_31,40,39.986887,27.902842
4,Block0106_2020_09_02,43,42.999997,19.608617
5,Block0106_2020_09_07,48,47.996502,22.709826
6,Block0106_2020_09_16,38,38.000000,8.905893


Block 0201

Predictions wth the frozen model

In [63]:
# first get the predictions
frozen_preds_block_0201 = get_all_preds_in_test_time_series('block_0201', 'All_data', frozen_model)

384/384 [==============================] - 2s 5ms/step


In [64]:
frozen_preds_block_0201.shape

(12288, 7)

In [65]:
frozen_final_forecasts_block_0201 = get_final_forecasted_and_true_values(frozen_preds_block_0201, image_height, image_width, 8, 32, 'true_counts_blk_0201.csv', 'frozen_block_0201_adjustment_1')

In [66]:
frozen_normalized_forecasts_block_0201 = frozen_final_forecasts_block_0201[0]

In [67]:
print(frozen_normalized_forecasts_block_0201)

[47.82192287199238, 60.90837893099585, 73.39188096729694, 57.49839280209077, 38.562896416018084, 32.09855127189186, 14.760988606036163]


In [68]:
mae_frozen_block_0201 = frozen_final_forecasts_block_0201[1]
mae_frozen_block_0201

[12.171162754061404,
 14.931940850838206,
 PearsonRResult(statistic=0.8469046944764594, pvalue=0.016191133783925785),
 -5.17241809100204]

In [69]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0201[2]
frozen_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0201_2020_08_26,45,45.000217,47.821923
1,Block0201_2020_08_27,45,45.000040,60.908379
2,Block0201_2020_08_28,47,47.000001,73.391881
3,Block0201_2020_08_31,38,38.000000,57.498393
4,Block0201_2020_09_02,42,42.000041,38.562896
5,Block0201_2020_09_07,35,35.000000,32.098551
6,Block0201_2020_09_16,29,29.000000,14.760989


Predictions with the finetuned model

In [70]:
# first get the predictions
finetuned_preds_block_0201 = get_all_preds_in_test_time_series('block_0201', 'All_data', finetuned_model)

384/384 [==============================] - 2s 5ms/step


In [71]:
finetuned_preds_block_0201.shape

(12288, 7)

In [72]:
finetuned_final_forecasts_block_0201 = get_final_forecasted_and_true_values(finetuned_preds_block_0201, image_height, image_width, 8, 32, 'true_counts_blk_0201.csv', 'finetuned_block_0201_adjustment_1')

In [73]:
finetuned_normalized_forecasts_block_0201 = finetuned_final_forecasts_block_0201[0]

In [74]:
print(finetuned_normalized_forecasts_block_0201)

[15.378535594964585, 32.89573440883093, 58.569914322139475, 31.719770124684448, 22.410719437257736, 25.989817520929137, 10.210935209943878]


In [75]:
mae_finetuned_block_0201 = finetuned_final_forecasts_block_0201[1]
mae_finetuned_block_0201

[15.280628860789822,
 16.967198862450992,
 PearsonRResult(statistic=0.5916203431846142, pvalue=0.1617464844344213),
 -6.969720917886982]

In [76]:
finetuned_true_forecasted_df = finetuned_final_forecasts_block_0201[2]
finetuned_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0201_2020_08_26,45,45.000217,15.378536
1,Block0201_2020_08_27,45,45.000040,32.895734
2,Block0201_2020_08_28,47,47.000001,58.569914
3,Block0201_2020_08_31,38,38.000000,31.719770
4,Block0201_2020_09_02,42,42.000041,22.410719
5,Block0201_2020_09_07,35,35.000000,25.989818
6,Block0201_2020_09_16,29,29.000000,10.210935


Block 0202

Predictions wth the frozen model

In [77]:
# first get the predictions
frozen_preds_block_0202 = get_all_preds_in_test_time_series('block_0202', 'All_data', frozen_model)

384/384 [==============================] - 2s 5ms/step


In [78]:
frozen_preds_block_0202.shape

(12288, 7)

In [79]:
frozen_final_forecasts_block_0202 = get_final_forecasted_and_true_values(frozen_preds_block_0202, image_height, image_width, 8, 32, 'true_counts_blk_0202.csv', 'frozen_block_0202_adjustment_1')

In [80]:
frozen_normalized_forecasts_block_0202 = frozen_final_forecasts_block_0202[0]

In [81]:
print(frozen_normalized_forecasts_block_0202)

[42.712468578468865, 58.437716580087184, 68.96296399161899, 55.737732263315095, 38.81640472260561, 30.17663961596437, 11.247731610181592]


In [82]:
mae_frozen_block_0202 = frozen_final_forecasts_block_0202[1]
mae_frozen_block_0202

[25.656599163125502,
 28.891127618923004,
 PearsonRResult(statistic=0.8225054825551005, pvalue=0.02311106222808),
 -242.45336606876126]

In [83]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0202[2]
frozen_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0202_2020_08_26,18,17.999960,42.712469
1,Block0202_2020_08_27,21,21.000000,58.437717
2,Block0202_2020_08_28,23,23.000000,68.962964
3,Block0202_2020_08_31,21,20.999982,55.737732
4,Block0202_2020_09_02,21,21.000000,38.816405
5,Block0202_2020_09_07,18,18.000000,30.176640
6,Block0202_2020_09_16,18,18.000000,11.247732


Predictions with the finetuned model

In [84]:
# first get the predictions
finetuned_preds_block_0202 = get_all_preds_in_test_time_series('block_0202', 'All_data', finetuned_model)

384/384 [==============================] - 2s 5ms/step


In [85]:
finetuned_preds_block_0202.shape

(12288, 7)

In [86]:
finetuned_final_forecasts_block_0202 = get_final_forecasted_and_true_values(finetuned_preds_block_0202, image_height, image_width, 8, 32, 'true_counts_blk_0202.csv', 'finetuned_block_0202_adjustment_1')

In [87]:
finetuned_normalized_forecasts_block_0202 = finetuned_final_forecasts_block_0202[0]

In [88]:
print(finetuned_normalized_forecasts_block_0202)

[14.896612084430066, 40.51698828403445, 64.99444861339485, 35.408512457843244, 26.20261955133568, 30.02829380847288, 9.456919614630957]


In [89]:
mae_finetuned_block_0202 = finetuned_final_forecasts_block_0202[1]
mae_finetuned_block_0202

[14.971047288002866,
 19.29623989439502,
 PearsonRResult(statistic=0.8575205983189446, pvalue=0.013609854716441097),
 -107.60058826809554]

In [90]:
finetuned_true_forecasted_df = finetuned_final_forecasts_block_0202[2]
finetuned_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0202_2020_08_26,18,17.999960,14.896612
1,Block0202_2020_08_27,21,21.000000,40.516988
2,Block0202_2020_08_28,23,23.000000,64.994449
3,Block0202_2020_08_31,21,20.999982,35.408512
4,Block0202_2020_09_02,21,21.000000,26.202620
5,Block0202_2020_09_07,18,18.000000,30.028294
6,Block0202_2020_09_16,18,18.000000,9.456920


Block 0205

Predictions wth the frozen model

In [91]:
# first get the predictions
frozen_preds_block_0205 = get_all_preds_in_test_time_series('block_0205', 'All_data', frozen_model)

384/384 [==============================] - 2s 5ms/step


In [92]:
frozen_preds_block_0205.shape

(12288, 7)

In [93]:
frozen_final_forecasts_block_0205 = get_final_forecasted_and_true_values(frozen_preds_block_0205, image_height, image_width, 8, 32, 'true_counts_blk_0205.csv', 'frozen_block_0205_adjustment_1')

In [94]:
frozen_normalized_forecasts_block_0205 = frozen_final_forecasts_block_0205[0]

In [95]:
print(frozen_normalized_forecasts_block_0205)

[43.3376891449827, 48.8251179432015, 61.51304990686041, 53.153190454599105, 32.57709436650322, 26.37388888609342, 14.216155443388033]


In [96]:
mae_frozen_block_0205 = frozen_final_forecasts_block_0205[1]
mae_frozen_block_0205

[10.426647209099091,
 11.917759584202736,
 PearsonRResult(statistic=0.87372420603894, pvalue=0.010156035470846817),
 -7.656239654024816]

In [97]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0205[2]
frozen_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0205_2020_08_26,44,44.000000,43.337689
1,Block0205_2020_08_27,42,42.000001,48.825118
2,Block0205_2020_08_28,45,45.000000,61.513050
3,Block0205_2020_08_31,43,43.000000,53.153190
4,Block0205_2020_09_02,39,39.000000,32.577094
5,Block0205_2020_09_07,41,40.999915,26.373889
6,Block0205_2020_09_16,32,31.999656,14.216155


Predictions with the finetuned model

In [98]:
# first get the predictions
finetuned_preds_block_0205 = get_all_preds_in_test_time_series('block_0205', 'All_data', finetuned_model)

384/384 [==============================] - 2s 5ms/step


In [99]:
finetuned_preds_block_0205.shape

(12288, 7)

In [100]:
finetuned_final_forecasts_block_0205 = get_final_forecasted_and_true_values(finetuned_preds_block_0205, image_height, image_width, 8, 32, 'true_counts_blk_0205.csv', 'finetuned_block_0205_adjustment_1')

In [101]:
finetuned_normalized_forecasts_block_0205 = finetuned_final_forecasts_block_0205[0]

In [102]:
print(finetuned_normalized_forecasts_block_0205)

[14.400292721436319, 28.965239256152547, 49.516024391623745, 27.442926516303586, 19.553090759729557, 22.622602299355222, 9.442773187820723]


In [103]:
mae_finetuned_block_0205 = finetuned_final_forecasts_block_0205[1]
mae_finetuned_block_0205

[17.584157092975115,
 19.023309675282782,
 PearsonRResult(statistic=0.6619752693997537, pvalue=0.10526525153499398),
 -21.055260247616502]

In [104]:
finetuned_true_forecasted_df = finetuned_final_forecasts_block_0205[2]
finetuned_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0205_2020_08_26,44,44.000000,14.400293
1,Block0205_2020_08_27,42,42.000001,28.965239
2,Block0205_2020_08_28,45,45.000000,49.516024
3,Block0205_2020_08_31,43,43.000000,27.442927
4,Block0205_2020_09_02,39,39.000000,19.553091
5,Block0205_2020_09_07,41,40.999915,22.622602
6,Block0205_2020_09_16,32,31.999656,9.442773


Block 0206

Predictions wth the frozen model

In [105]:
# first get the predictions
frozen_preds_block_0206 = get_all_preds_in_test_time_series('block_0206', 'All_data', frozen_model)

384/384 [==============================] - 2s 5ms/step


In [106]:
frozen_preds_block_0206.shape

(12288, 7)

In [107]:
frozen_final_forecasts_block_0206 = get_final_forecasted_and_true_values(frozen_preds_block_0206, image_height, image_width, 8, 32, 'true_counts_blk_0206.csv', 'frozen_block_0206_adjustment_1')

In [108]:
frozen_normalized_forecasts_block_0206 = frozen_final_forecasts_block_0206[0]

In [109]:
print(frozen_normalized_forecasts_block_0206)

[46.68564219171417, 56.45632996016767, 68.57609160753215, 54.64241671139218, 34.776878446636395, 28.272996986926145, 13.346776661940064]


In [110]:
mae_frozen_block_0206 = frozen_final_forecasts_block_0206[1]
mae_frozen_block_0206

[13.151939891775523,
 15.918455457995057,
 PearsonRResult(statistic=0.8739188085754841, pvalue=0.010118054356453),
 -2.203422080557394]

In [111]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0206[2]
frozen_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0206_2020_08_26,41,40.999838,46.685642
1,Block0206_2020_08_27,42,41.998810,56.456330
2,Block0206_2020_08_28,39,39.000067,68.576092
3,Block0206_2020_08_31,32,32.000003,54.642417
4,Block0206_2020_09_02,25,25.000352,34.776878
5,Block0206_2020_09_07,23,23.000040,28.272997
6,Block0206_2020_09_16,18,18.000000,13.346777


Predictions with the finetuned model

In [112]:
# first get the predictions
finetuned_preds_block_0206 = get_all_preds_in_test_time_series('block_0206', 'All_data', finetuned_model)

384/384 [==============================] - 2s 5ms/step


In [113]:
finetuned_preds_block_0206.shape

(12288, 7)

In [114]:
finetuned_final_forecasts_block_0206 = get_final_forecasted_and_true_values(finetuned_preds_block_0206, image_height, image_width, 8, 32, 'true_counts_blk_0206.csv', 'finetuned_block_0206_adjustment_1')

In [115]:
finetuned_normalized_forecasts_block_0206 = finetuned_final_forecasts_block_0206[0]

In [116]:
print(finetuned_normalized_forecasts_block_0206)

[15.867842902214457, 33.37555232479886, 57.79342410764203, 31.55582802114214, 22.681228372322643, 26.265741958375884, 10.321021995721821]


In [117]:
mae_finetuned_block_0206 = finetuned_final_forecasts_block_0206[1]
mae_finetuned_block_0206

[9.465384635688284,
 12.730194545235266,
 PearsonRResult(statistic=0.5126442809415626, pvalue=0.2394049509295216),
 -1.0487189898909572]

In [118]:
finetuned_true_forecasted_df = finetuned_final_forecasts_block_0206[2]
finetuned_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0206_2020_08_26,41,40.999838,15.867843
1,Block0206_2020_08_27,42,41.998810,33.375552
2,Block0206_2020_08_28,39,39.000067,57.793424
3,Block0206_2020_08_31,32,32.000003,31.555828
4,Block0206_2020_09_02,25,25.000352,22.681228
5,Block0206_2020_09_07,23,23.000040,26.265742
6,Block0206_2020_09_16,18,18.000000,10.321022


Block 0302

Predictions wth the frozen model

In [119]:
# first get the predictions
frozen_preds_block_0302 = get_all_preds_in_test_time_series('block_0302', 'All_data', frozen_model)

384/384 [==============================] - 2s 5ms/step


In [120]:
frozen_preds_block_0302.shape

(12288, 7)

In [121]:
frozen_final_forecasts_block_0302 = get_final_forecasted_and_true_values(frozen_preds_block_0302, image_height, image_width, 8, 32, 'true_counts_blk_0302.csv', 'frozen_block_0302_adjustment_1')

In [122]:
frozen_normalized_forecasts_block_0302 = frozen_final_forecasts_block_0302[0]

In [123]:
print(frozen_normalized_forecasts_block_0302)

[48.80199099044854, 61.919243021500534, 74.11613898133915, 58.84052458723535, 40.54594253631867, 32.765373438534986, 15.619518509446653]


In [124]:
mae_frozen_block_0302 = frozen_final_forecasts_block_0302[1]
mae_frozen_block_0302

[11.591868730760883,
 13.861566617196107,
 PearsonRResult(statistic=0.8944578050854662, pvalue=0.006561624257505954),
 -6.355475332082275]

In [125]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0302[2]
frozen_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0302_2020_08_26,49,49.000000,48.801991
1,Block0302_2020_08_27,49,49.000005,61.919243
2,Block0302_2020_08_28,54,53.999570,74.116139
3,Block0302_2020_08_31,50,50.042349,58.840525
4,Block0302_2020_09_02,43,43.000009,40.545943
5,Block0302_2020_09_07,48,48.000572,32.765373
6,Block0302_2020_09_16,37,36.999999,15.619519


Predictions with the finetuned model

In [126]:
# first get the predictions
finetuned_preds_block_0302 = get_all_preds_in_test_time_series('block_0302', 'All_data', finetuned_model)

384/384 [==============================] - 2s 5ms/step


In [127]:
finetuned_preds_block_0302.shape

(12288, 7)

In [128]:
finetuned_final_forecasts_block_0302 = get_final_forecasted_and_true_values(finetuned_preds_block_0302, image_height, image_width, 8, 32, 'true_counts_blk_0302.csv', 'finetuned_block_0302_adjustment_1')

In [129]:
finetuned_normalized_forecasts_block_0302 = finetuned_final_forecasts_block_0302[0]

In [130]:
print(finetuned_normalized_forecasts_block_0302)

[16.41342652238866, 34.66610831923583, 58.18659104292035, 32.5557020413276, 23.3366132743364, 26.973166602290696, 10.627195832225274]


In [131]:
mae_finetuned_block_0302 = finetuned_final_forecasts_block_0302[1]
mae_finetuned_block_0302

[19.373482635873696,
 21.08986756542131,
 PearsonRResult(statistic=0.7933745247279083, pvalue=0.03323272448338166),
 -16.026830611268345]

In [132]:
finetuned_true_forecasted_df = finetuned_final_forecasts_block_0302[2]
finetuned_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0302_2020_08_26,49,49.000000,16.413427
1,Block0302_2020_08_27,49,49.000005,34.666108
2,Block0302_2020_08_28,54,53.999570,58.186591
3,Block0302_2020_08_31,50,50.042349,32.555702
4,Block0302_2020_09_02,43,43.000009,23.336613
5,Block0302_2020_09_07,48,48.000572,26.973167
6,Block0302_2020_09_16,37,36.999999,10.627196


Block 0303

Predictions wth the frozen model

In [133]:
# first get the predictions
frozen_preds_block_0303 = get_all_preds_in_test_time_series('block_0303', 'All_data', frozen_model)

384/384 [==============================] - 2s 5ms/step


In [134]:
frozen_preds_block_0303.shape

(12288, 7)

In [135]:
frozen_final_forecasts_block_0303 = get_final_forecasted_and_true_values(frozen_preds_block_0303, image_height, image_width, 8, 32, 'true_counts_blk_0303.csv', 'frozen_block_0303_adjustment_1')

In [136]:
frozen_normalized_forecasts_block_0303 = frozen_final_forecasts_block_0303[0]

In [137]:
print(frozen_normalized_forecasts_block_0303)

[46.04724565916048, 59.078410864475075, 72.24396866062426, 57.60904245750215, 38.86255802266898, 31.802889790053285, 14.961049264855271]


In [138]:
mae_frozen_block_0303 = frozen_final_forecasts_block_0303[1]
mae_frozen_block_0303

[11.283256470171631,
 14.499775636765296,
 PearsonRResult(statistic=0.7690630103989092, pvalue=0.04327422223160038),
 -2.353493223408232]

In [139]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0303[2]
frozen_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0303_2020_08_26,49,49.000171,46.047246
1,Block0303_2020_08_27,46,46.000007,59.078411
2,Block0303_2020_08_28,43,42.999982,72.243969
3,Block0303_2020_08_31,39,38.999993,57.609042
4,Block0303_2020_09_02,36,36.025884,38.862558
5,Block0303_2020_09_07,36,36.000000,31.802890
6,Block0303_2020_09_16,23,23.000000,14.961049


Predictions with the finetuned model

In [140]:
# first get the predictions
finetuned_preds_block_0303 = get_all_preds_in_test_time_series('block_0303', 'All_data', finetuned_model)

384/384 [==============================] - 2s 5ms/step


In [141]:
finetuned_preds_block_0303.shape

(12288, 7)

In [142]:
finetuned_final_forecasts_block_0303 = get_final_forecasted_and_true_values(finetuned_preds_block_0303, image_height, image_width, 8, 32, 'true_counts_blk_0303.csv', 'finetuned_block_0303_adjustment_1')

In [143]:
finetuned_normalized_forecasts_block_0303 = finetuned_final_forecasts_block_0303[0]

In [144]:
print(finetuned_normalized_forecasts_block_0303)

[13.217576595702365, 28.85109958193223, 53.095997570367906, 27.546840267165663, 19.653354678833235, 23.11470795013176, 8.296113263591703]


In [145]:
mae_finetuned_block_0303 = finetuned_final_forecasts_block_0303[1]
mae_finetuned_block_0303

[16.91661503328728,
 18.73506450735575,
 PearsonRResult(statistic=0.4172092373348355, pvalue=0.35172151784035616),
 -4.598674955287852]

In [146]:
finetuned_true_forecasted_df = finetuned_final_forecasts_block_0303[2]
finetuned_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0303_2020_08_26,49,49.000171,13.217577
1,Block0303_2020_08_27,46,46.000007,28.851100
2,Block0303_2020_08_28,43,42.999982,53.095998
3,Block0303_2020_08_31,39,38.999993,27.546840
4,Block0303_2020_09_02,36,36.025884,19.653355
5,Block0303_2020_09_07,36,36.000000,23.114708
6,Block0303_2020_09_16,23,23.000000,8.296113


Block 0304

Predictions wth the frozen model

In [147]:
# first get the predictions
frozen_preds_block_0304 = get_all_preds_in_test_time_series('block_0304', 'All_data', frozen_model)

384/384 [==============================] - 2s 5ms/step


In [148]:
frozen_preds_block_0304.shape

(12288, 7)

In [149]:
frozen_final_forecasts_block_0304 = get_final_forecasted_and_true_values(frozen_preds_block_0304, image_height, image_width, 8, 32, 'true_counts_blk_0304.csv', 'frozen_block_0304')

In [150]:
frozen_normalized_forecasts_block_0304 = frozen_final_forecasts_block_0304[0]

In [151]:
print(frozen_normalized_forecasts_block_0304)

[44.442850482769735, 52.08201234152527, 64.8549644943398, 55.46065248647936, 35.25931413911167, 28.779722967463485, 14.22855783150761]


In [152]:
mae_frozen_block_0304 = frozen_final_forecasts_block_0304[1]
mae_frozen_block_0304

[10.22469783814734,
 11.755116774007655,
 PearsonRResult(statistic=0.9402707599679868, pvalue=0.0016213670871656383),
 -2.7784351273198946]

In [153]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0304[2]
frozen_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0304_2020_08_26,37,37.000000,44.442850
1,Block0304_2020_08_27,41,41.002057,52.082012
2,Block0304_2020_08_28,43,42.999998,64.854964
3,Block0304_2020_08_31,42,41.999955,55.460652
4,Block0304_2020_09_02,38,38.000000,35.259314
5,Block0304_2020_09_07,34,34.000000,28.779723
6,Block0304_2020_09_16,24,24.000000,14.228558


Predictions with the finetuned model

In [154]:
# first get the predictions
finetuned_preds_block_0304 = get_all_preds_in_test_time_series('block_0304', 'All_data', finetuned_model)

384/384 [==============================] - 2s 5ms/step


In [155]:
finetuned_preds_block_0304.shape

(12288, 7)

In [156]:
finetuned_final_forecasts_block_0304 = get_final_forecasted_and_true_values(finetuned_preds_block_0304, image_height, image_width, 8, 32, 'true_counts_blk_0304.csv', 'finetuned_block_0304')

In [157]:
finetuned_normalized_forecasts_block_0304 = finetuned_final_forecasts_block_0304[0]

In [158]:
print(finetuned_normalized_forecasts_block_0304)

[13.10236493316491, 27.384856162374092, 51.92208062586966, 26.24075369698871, 18.68544235651027, 22.07676898203883, 8.073458982992078]


In [159]:
mae_finetuned_block_0304 = finetuned_final_forecasts_block_0304[1]
mae_finetuned_block_0304

[15.622633644542969,
 16.27134533852229,
 PearsonRResult(statistic=0.7211513543436602, pvalue=0.06740822596269384),
 -6.239440444836553]

In [160]:
finetuned_true_forecasted_df = finetuned_final_forecasts_block_0304[2]
finetuned_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0304_2020_08_26,37,37.000000,13.102365
1,Block0304_2020_08_27,41,41.002057,27.384856
2,Block0304_2020_08_28,43,42.999998,51.922081
3,Block0304_2020_08_31,42,41.999955,26.240754
4,Block0304_2020_09_02,38,38.000000,18.685442
5,Block0304_2020_09_07,34,34.000000,22.076769
6,Block0304_2020_09_16,24,24.000000,8.073459


Block 0305

Predictions wth the frozen model

In [161]:
# first get the predictions
frozen_preds_block_0305 = get_all_preds_in_test_time_series('block_0305', 'All_data', frozen_model)

384/384 [==============================] - 2s 5ms/step


In [162]:
frozen_preds_block_0305.shape

(12288, 7)

In [163]:
frozen_final_forecasts_block_0305 = get_final_forecasted_and_true_values(frozen_preds_block_0305, image_height, image_width, 8, 32, 'true_counts_blk_0305.csv', 'frozen_block_0305_adjustment_1')

In [164]:
frozen_normalized_forecasts_block_0305 = frozen_final_forecasts_block_0305[0]

In [165]:
print(frozen_normalized_forecasts_block_0305)

[48.655040854151615, 58.97871193287767, 69.8038033012041, 55.683731919568416, 37.44218583326195, 29.910789008811268, 13.749447661040303]


In [166]:
mae_frozen_block_0305 = frozen_final_forecasts_block_0305[1]
mae_frozen_block_0305

[13.532116455547818,
 18.375584255210274,
 PearsonRResult(statistic=0.6442198145503649, pvalue=0.11835501500349038),
 -5.109838530020774]

In [167]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0305[2]
frozen_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0305_2020_08_26,46,46.000000,48.655041
1,Block0305_2020_08_27,35,35.000000,58.978712
2,Block0305_2020_08_28,37,37.000000,69.803803
3,Block0305_2020_08_31,30,30.000652,55.683732
4,Block0305_2020_09_02,35,35.001190,37.442186
5,Block0305_2020_09_07,29,28.999994,29.910789
6,Block0305_2020_09_16,20,20.000018,13.749448


Predictions with the finetuned model

In [168]:
# first get the predictions
finetuned_preds_block_0305 = get_all_preds_in_test_time_series('block_0305', 'All_data', finetuned_model)

384/384 [==============================] - 2s 5ms/step


In [169]:
finetuned_preds_block_0305.shape

(12288, 7)

In [170]:
finetuned_final_forecasts_block_0305 = get_final_forecasted_and_true_values(finetuned_preds_block_0305, image_height, image_width, 8, 32, 'true_counts_blk_0305.csv', 'finetuned_block_0305_adjustment_1')

In [171]:
finetuned_normalized_forecasts_block_0305 = finetuned_final_forecasts_block_0305[0]

In [172]:
print(finetuned_normalized_forecasts_block_0305)

[17.437471993477736, 38.08966047763377, 62.418232353972506, 34.5591313584955, 25.241523874989518, 28.927370249997495, 10.903414521437902]


In [173]:
mae_finetuned_block_0305 = finetuned_final_forecasts_block_0305[1]
mae_finetuned_block_0305

[11.508177650028447,
 15.44676212704575,
 PearsonRResult(statistic=0.26186192171750117, pvalue=0.5705331031602254),
 -3.31740049862157]

In [174]:
finetuned_true_forecasted_df = finetuned_final_forecasts_block_0305[2]
finetuned_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0305_2020_08_26,46,46.000000,17.437472
1,Block0305_2020_08_27,35,35.000000,38.089660
2,Block0305_2020_08_28,37,37.000000,62.418232
3,Block0305_2020_08_31,30,30.000652,34.559131
4,Block0305_2020_09_02,35,35.001190,25.241524
5,Block0305_2020_09_07,29,28.999994,28.927370
6,Block0305_2020_09_16,20,20.000018,10.903415


Block 0306

Predictions wth the frozen model

In [175]:
# first get the predictions
frozen_preds_block_0306 = get_all_preds_in_test_time_series('block_0306', 'All_data', frozen_model)

384/384 [==============================] - 2s 5ms/step


In [176]:
frozen_preds_block_0306.shape

(12288, 7)

In [177]:
frozen_final_forecasts_block_0306 = get_final_forecasted_and_true_values(frozen_preds_block_0306, image_height, image_width, 8, 32, 'true_counts_blk_0306.csv', 'frozen_block_0306_adjustment_1')

In [178]:
frozen_normalized_forecasts_block_0306 = frozen_final_forecasts_block_0306[0]

In [179]:
print(frozen_normalized_forecasts_block_0306)

[47.462667648267804, 54.5701207562775, 66.54666048162865, 56.48388929318432, 36.10900274971926, 29.15743204937793, 15.14996746728232]


In [180]:
mae_frozen_block_0306 = frozen_final_forecasts_block_0306[1]
mae_frozen_block_0306

[10.092419416139823,
 12.479159484249609,
 PearsonRResult(statistic=0.8661188589969857, pvalue=0.01170498882169279),
 -1.360996797720765]

In [181]:
frozen_true_forecasted_df = frozen_final_forecasts_block_0306[2]
frozen_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0306_2020_08_26,41,41.000009,47.462668
1,Block0306_2020_08_27,41,41.000003,54.570121
2,Block0306_2020_08_28,43,43.006851,66.546660
3,Block0306_2020_08_31,40,39.997604,56.483889
4,Block0306_2020_09_02,40,40.000000,36.109003
5,Block0306_2020_09_07,33,32.999982,29.157432
6,Block0306_2020_09_16,18,18.000000,15.149967


Predictions with the finetuned model

In [182]:
# first get the predictions
finetuned_preds_block_0306 = get_all_preds_in_test_time_series('block_0306', 'All_data', finetuned_model)

384/384 [==============================] - 2s 5ms/step


In [183]:
finetuned_preds_block_0306.shape

(12288, 7)

In [184]:
finetuned_final_forecasts_block_0306 = get_final_forecasted_and_true_values(finetuned_preds_block_0306, image_height, image_width, 8, 32, 'true_counts_blk_0306.csv', 'finetuned_block_0306_adjustment_1')

In [185]:
finetuned_normalized_forecasts_block_0306 = finetuned_final_forecasts_block_0306[0]

In [186]:
print(finetuned_normalized_forecasts_block_0306)

[16.325208342086135, 32.96739017065055, 56.3641078187599, 32.4122765810383, 23.168542981403036, 26.797587541447832, 10.613592354232727]


In [187]:
mae_finetuned_block_0306 = finetuned_final_forecasts_block_0306[1]
mae_finetuned_block_0306

[12.011358549700189,
 13.553428517698,
 PearsonRResult(statistic=0.6148681959273782, pvalue=0.14172499770403882),
 -1.7849863256909413]

In [188]:
finetuned_true_forecasted_df = finetuned_final_forecasts_block_0306[2]
finetuned_true_forecasted_df

,Image_name,True_count,Convolved_count,Forecasted_value
0,Block0306_2020_08_26,41,41.000009,16.325208
1,Block0306_2020_08_27,41,41.000003,32.967390
2,Block0306_2020_08_28,43,43.006851,56.364108
3,Block0306_2020_08_31,40,39.997604,32.412277
4,Block0306_2020_09_02,40,40.000000,23.168543
5,Block0306_2020_09_07,33,32.999982,26.797588
6,Block0306_2020_09_16,18,18.000000,10.613592
